Predicting Customer Churn

This project uses Machine Learning to predict "Customer Churn"—which is when telecom customers cancel their service. Since finding new customers is much more expensive than keeping existing ones, our goal is to identify unhappy users before they leave so the company can save money. We analyzed historical data from 7,043 customers and trained five different AI models to predict if a user will stay or go.

In [ ]:
#Imports
#Below are all the imports used in the notebook.
# Common
import os
import numpy as np
import pandas as pd

# Data Visualization
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

# Data Processing
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split

# ML Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ANN
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras import callbacks

# Performace Measures
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report


In [ ]:
file_path = 'data/Customer-Churn.csv'

df = pd.read_csv(file_path)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
#Calculate the Num of Colums
cols = df.columns
n_cols = len(cols)
print(cols)
# Print
print(f"Total Number of Columns : {n_cols}")

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')
Total Number of Columns : 21


In [ ]:
#We have 21 columns, out of which 20 are Feature Columns and one is the Target Column.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Initialize the encoder
le = LabelEncoder()

# 1. First, fix 'TotalCharges' to be a number (so we don't accidentally encode it)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# 2. Drop 'customerID' if it is still there (it messes up training)
if 'customerID' in df.columns:
    df.drop(columns=['customerID'], inplace=True)

# 3. Loop through all remaining columns that are type 'object'
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])
    print(f"Encoded {col}")


Encoded gender
Encoded Partner
Encoded Dependents
Encoded PhoneService
Encoded MultipleLines
Encoded InternetService
Encoded OnlineSecurity
Encoded OnlineBackup
Encoded DeviceProtection
Encoded TechSupport
Encoded StreamingTV
Encoded StreamingMovies
Encoded Contract
Encoded PaperlessBilling
Encoded PaymentMethod
Encoded Churn


In [ ]:
# Check the final result
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   int64  
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   int64  
 3   Dependents        7043 non-null   int64  
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   int64  
 6   MultipleLines     7043 non-null   int64  
 7   InternetService   7043 non-null   int64  
 8   OnlineSecurity    7043 non-null   int64  
 9   OnlineBackup      7043 non-null   int64  
 10  DeviceProtection  7043 non-null   int64  
 11  TechSupport       7043 non-null   int64  
 12  StreamingTV       7043 non-null   int64  
 13  StreamingMovies   7043 non-null   int64  
 14  Contract          7043 non-null   int64  
 15  PaperlessBilling  7043 non-null   int64  
 16  PaymentMethod     7043 non-null   int64  


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65


In [ ]:
print(df.Churn.value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [ ]:
# Data Splitting
# Seperate Features and Columns
y = df.pop('Churn').to_numpy()
X = df.to_numpy()

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# Split into Training and Testing

X_train,X_test,y_train,y_test=train_test_split(X_scaled,y,test_size=0.25,random_state=42,shuffle=True)

In [ ]:
# ML Models

# Logistic regression
lr = LogisticRegression()
lr.fit(X_train, y_train)

# Prediction
pred = lr.predict(X_test)

# Calc  confusionmatrix
cm = confusion_matrix(pred, y_test)

# Show CM and classification_repor
print("Logistic Regression : \n")
print(f"\nConfusion Matrix : \n")
fig = px.imshow(cm, text_auto=True, color_continuous_scale='fall', width=400, height=400)
fig.show()
print(classification_report(y_test, pred))

Logistic Regression : 


Confusion Matrix : 



              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1282
           1       0.68      0.56      0.62       479

    accuracy                           0.81      1761
   macro avg       0.77      0.73      0.75      1761
weighted avg       0.80      0.81      0.80      1761



In [ ]:
# Decision Tree Classifier
dtc = DecisionTreeClassifier()
dtc.fit(X_train, y_train)

# Prediction
pred = dtc.predict(X_test)

# Calc  confusion matrix
cm = confusion_matrix(pred, y_test)

# Show CM and classification_repor
print("Decision Tree Classifier : \n")
print(f"\nConfusion Matrix :")
fig = px.imshow(cm, text_auto=True, color_continuous_scale='fall', width=400, height=400)
fig.show()
print(classification_report(y_test, pred))


Decision Tree Classifier : 


Confusion Matrix :


              precision    recall  f1-score   support

           0       0.81      0.83      0.82      1282
           1       0.51      0.49      0.50       479

    accuracy                           0.73      1761
   macro avg       0.66      0.66      0.66      1761
weighted avg       0.73      0.73      0.73      1761



In [ ]:
# Random Forest Classifier
RF = RandomForestClassifier(max_depth=4)
RF.fit(X_train, y_train)

# Prediction
pred = RF.predict(X_test)

# Calc  confusion matrix
cm = confusion_matrix(pred, y_test)

# Show CM and classification_repor
print("Random Forest Classifier : \n")
print(f"\nConfusion Matrix :")
fig = px.imshow(cm, text_auto=True, color_continuous_scale='fall', width=400, height=400)
fig.show()
print(classification_report(y_test, pred))

Random Forest Classifier : 


Confusion Matrix :


              precision    recall  f1-score   support

           0       0.81      0.93      0.86      1282
           1       0.68      0.41      0.51       479

    accuracy                           0.79      1761
   macro avg       0.74      0.67      0.69      1761
weighted avg       0.77      0.79      0.77      1761



In [ ]:
# XGB Classifier
XGB = XGBClassifier(max_depth=6)
XGB.fit(X_train, y_train)

# Prediction
pred = XGB.predict(X_test)

# Calc  confusion matrix
cm = confusion_matrix(pred, y_test)

# Show CM and classification_repor
print("XGB Classifier : \n")
print(f"\nConfusion Matrix :")
fig = px.imshow(cm, text_auto=True, color_continuous_scale='fall', width=400, height=400)
fig.show()
print(classification_report(y_test, pred))

XGB Classifier : 


Confusion Matrix :


              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1282
           1       0.63      0.54      0.58       479

    accuracy                           0.79      1761
   macro avg       0.74      0.71      0.72      1761
weighted avg       0.78      0.79      0.78      1761



In [ ]:
# ANN Classifier
model=Sequential()
model.add(Dense(16, activation='relu')) # First hidden layer
model.add(Dropout(0.25))
model.add(Dense(16, activation='relu')) # Second hidden layer
model.add(Dropout(0.25))
model.add(Dense(1, activation='sigmoid')) # Output layer

In [ ]:
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['Accuracy'])

In [ ]:
earlystopping = callbacks.EarlyStopping(monitor='val_loss',
                                        mode='max',
                                        verbose=1,
                                        patience=20)

In [ ]:
history = model.fit(X_train, y_train,validation_split=0.1, batch_size = 32, epochs = 500, callbacks =[earlystopping])

Epoch 1/500
149/149 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - Accuracy: 0.5277 - loss: 0.7096 - val_Accuracy: 0.7656 - val_loss: 0.4959
Epoch 2/500
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Accuracy: 0.7303 - loss: 0.5273 - val_Accuracy: 0.7694 - val_loss: 0.4491
Epoch 3/500
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Accuracy: 0.7436 - loss: 0.4926 - val_Accuracy: 0.7883 - val_loss: 0.4358
Epoch 4/500
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - Accuracy: 0.7664 - loss: 0.4597 - val_Accuracy: 0.7940 - val_loss: 0.4298
Epoch 5/500
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - Accuracy: 0.7780 - loss: 0.4514 - val_Accuracy: 0.7845 - val_loss: 0.4284
Epoch 6/500
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - Accuracy: 0.7618 - loss: 0.4615 - val_Accuracy: 0.7902 - val_loss: 0.4274
Epoch 7/500
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - Accuracy: 0.7795 - loss: 0.4534 - val_Accuracy: 0.7940 - val_loss: 0.4257
Epoch 8/500
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - Accuracy: 0.7778 - loss: 0.4505 - val_Acc

In [ ]:
# Prediction
y_pred = model.predict(X_test)
pred = (y_pred > 0.5)

# Calc  confusion matrix
cm = confusion_matrix(pred, y_test)

# Show CM and classification_repor
print("ANN Classifier : \n")
print(f"\nConfusion Matrix :")
fig = px.imshow(cm, text_auto=True, color_continuous_scale='fall', width=400, height=400)
fig.show()
print(classification_report(y_test, pred))

56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
ANN Classifier : 


Confusion Matrix :


              precision    recall  f1-score   support

           0       0.83      0.92      0.87      1282
           1       0.70      0.49      0.58       479

    accuracy                           0.80      1761
   macro avg       0.76      0.71      0.73      1761
weighted avg       0.79      0.80      0.79      1761

